In [ ]:
# library imports
import pandas as pd                             # tables, dataframes
import numpy as np                              # arrays, numerical computations
import matplotlib.pyplot as plt                 # plots
from uncertainties import ufloat as uf          # uncertainty propagation
from scipy.optimize import curve_fit            # fitting

from pathlib import Path                        # paths in operating system
# path config
downloads_dir = Path.home() / 'Downloads'
# make sure the folder exists (just in case)
downloads_dir.mkdir(parents=True, exist_ok=True)
# output_path = downloads_dir / 'my_plot.pdf'

In [ ]:
# custom functions

# type B uncertainty for digital instruments
def unc_B_digital(reading, percent_reading, digits, resolution):    # everything must be in the correct units; works for a single set of conditions only
    '''
    Type B uncertainty for digital measurements (the limiting/maximum error is kept)

    reading : float or array
        measured value
    percent_reading : float
        % of reading
    digits : float
        number of digits
    resolution : float
        smallest division (least significant digit)
    return : float or array
        type B uncertainty
    '''
    max_error = abs(reading) * (percent_reading/100) + digits * resolution  # max_error -- limiting (maximum) error
    u_B = max_error # divide by (3**0.5) for standard error

    return u_B

# type A uncertainty
def unc_A(data):
    '''
    Type A uncertainty (standard error of the mean)

    data : list or numpy array
        measurements
    return : float
        type A uncertainty
    '''
    data = np.array(data)
    n = len(data)

    s = np.std(data, ddof=1)
    u_A = s / np.sqrt(n)

    return u_A

# conversion from popt and pcov to ufloats
from uncertainties import correlated_values     # popt, pcov, correlations
def popt_pcov_to_ufloat(popt, pcov, correlated=False):
    '''
    Converts the result of curve_fit (popt, pcov) to ufloats.

    popt : array_like
        optimal parameters from curve_fit
    pcov : array_like
        covariance matrix from curve_fit
    correlated : bool
        True  -> ufloats carry correlations between parameters
        False -> independent ufloats (only sqrt of the pcov diagonal)
    return : tuple(ufloat, ...)
        parameters with uncertainties
    '''
    popt = np.asarray(popt, dtype=float)
    pcov = np.asarray(pcov, dtype=float)

    if correlated:
        return tuple(correlated_values(popt, pcov))

    perr = np.sqrt(np.diag(pcov))

    return tuple(uf(v, e) for v, e in zip(popt, perr))

# uarray to LaTeX
def uarray_to_latex(*array, digit=1, format='u', style='L', only_return=False):
    '''
    Formats ufloats as LaTeX strings (e.g. $1.23 \\pm 0.04$)

    *array : ufloat
        one or more ufloats to format
    digit : int
        number of digits / precision in the format specification
    format : str
        ufloat format type (e.g. 'u' for the uncertainty in parentheses)
    style : str
        output style; 'L' for LaTeX
    only_return : bool
        True  -> return the list of strings
        False -> print each string
    return : list[str] or None
        formatted strings if only_return=True, otherwise None
    '''
    output = [f'${u:.{digit}{format}{style}}$' for u in array]
    if only_return:
        return output
    else:
        for s in output:
            print(s)

In [ ]:
# nacteni google tabulek
sheet_id = '1UsP814IJrfRsE0t9Es3ti1vHPey5zNOwflnBbbTUxkY'
gid = '312593068'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'
df_impedance = pd.read_csv(url)

In [ ]:
# impedance


Ri = df_impedance['Ri'].iloc[0]
odpor_u20 = np.array(df_impedance['odpor_U20'])*1e-3
odpor_i = odpor_u20 / Ri
odpor_um0 = np.array(df_impedance['odpor_UM0'])*1e-3
odpor_rr = odpor_um0 / odpor_i
unc_rr = unc_A(odpor_rr)
prumer_odpor_rr = uf(np.mean(odpor_rr), unc_rr)
print(f'RR = {np.array2string(odpor_rr, precision=2)}\nR_R = {prumer_odpor_rr:.1u}')

kond_u20 = np.array(df_impedance['kond_U20'])*1e-3
kond_i = kond_u20 / Ri
kond_um0 = np.array(df_impedance['kond_UM0'])
kond_z = kond_um0 / kond_i
kond_f = np.array(df_impedance['kond_f'])
kond_phi = np.array(df_impedance['kond_phiM2'])*(np.pi/180)
kond_c = -1/(2*np.pi*kond_f*kond_z*np.sin(kond_phi))
kond_rc = kond_z * np.cos(kond_phi)
prumer_kond_c = uf(np.mean(kond_c), unc_A(kond_c))
print(f'ZC = {np.array2string(kond_z, precision=1)},\nC  = {np.array2string(kond_c, precision=2)},\nRC = {np.array2string(kond_rc, precision=2)}\nC_C = {prumer_kond_c:.1u}')

civka_u20 = np.array(df_impedance['civka_U20'])*1e-3
civka_i = civka_u20 / Ri
civka_um0 = np.array(df_impedance['civka_UM0'])
civka_z = civka_um0 / civka_i
civka_f = np.array(df_impedance['civka_f'])
civka_phi = np.array(df_impedance['civka_phiM2'])*(np.pi/180)
civka_l = (civka_z*np.sin(civka_phi))/(2*np.pi*civka_f)
civka_rl = civka_z*np.cos(civka_phi)
prumer_civka_l = uf(np.mean(civka_l), unc_A(civka_l))
print(f'ZL = {np.array2string(civka_z, precision=2)},\nL  = {np.array2string(civka_l, precision=4)},\nRL = {np.array2string(civka_rl, precision=2)}\nL_L = {prumer_civka_l:.1u}')

In [ ]:
# nacteni google tabulek
sheet_id = '1UsP814IJrfRsE0t9Es3ti1vHPey5zNOwflnBbbTUxkY'
gid = '800069381'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'
df_rezonance = pd.read_csv(url)

In [ ]:
# vypocet bez fitu

f0_guess = uf(1014, 4)
UM0 = uf(801, 6)
U20 = uf(210.3, 0.5)
RI_guess = 10.4
phim2 = uf(0.2, 8)

Gf0 = U20/(RI_guess*UM0)
uarray_to_latex(Gf0*1000)
R_guess = 1/Gf0
uarray_to_latex(R_guess)
G_fwhm_guess = Gf0/(2**0.5)
uarray_to_latex(G_fwhm_guess*1000)

f1_guess = uf(985, 3)
f2_guess = uf(1040, 2)
alpha_guess = np.pi*(f2_guess - f1_guess)
uarray_to_latex(alpha_guess)
L_guess = R_guess/(2*alpha_guess)
uarray_to_latex(L_guess)
C_guess = 1/((2*np.pi*f0_guess)**2 * L_guess)
uarray_to_latex(C_guess)

In [ ]:
# rezonance
rez_f = np.array(df_rezonance['f'])
rez_um0 = np.array(df_rezonance['UM0'])*1e-3 # hodnoty vetsi nez 1000 byly namereny v rezimu 9.999 Volt
rez_u20 = np.array(df_rezonance['U20'])*1e-3
rez_phi = np.array(df_rezonance['phiM2'])*(np.pi/180)
rez_g = rez_u20/(Ri*rez_um0)
rez_phi_g = -rez_phi
print(f'G = {rez_g}\nphi_g = {rez_phi_g*180/np.pi}')
print(1/rez_g[9])
print(rez_g[9]/(2**0.5))

alapha_odhad_fwhm = np.pi*(1040 - 985)
L_odhad_tabulka = (1/rez_g[9])/(2*alapha_odhad_fwhm)
C_odhad_tabulka = 1/(((2*np.pi*rez_f[9])**2)*L_odhad_tabulka)
print(C_odhad_tabulka*1e9)

In [ ]:
# fit a graf vodivosti abs(G)

def model_G(f, R, L, C):
    return 1/((R**2 + (2*np.pi*f*L - 1/(2*np.pi*f*C))**2)**0.5)
p0_G = [30, 113*1e-3, 217*1e-9]
popt_G, pcov_G = curve_fit(model_G, rez_f, rez_g, p0_G)
R, L, C = popt_pcov_to_ufloat(popt_G, pcov_G)
F = 1/L
Q = (1/R)*((L/C)**0.5)
alpha = R/(2*L)
omega0 = 1/((L*C)**0.5)
f0 =  omega0/(2*np.pi)
print(f'R = {R:.1uPL}\nL = {L:.1uPL}\nC = {C:.1uPL}\nF = {F:.1uPL}\nQ = {Q:.1uPL}\nalpha = {alpha:.1uPL}\nomega0 = {omega0:.1uPL}\nf0 = {f0:.1uPL}')

linspace_f = np.linspace(min(rez_f), max(rez_f), 1000)
def model_phi(f, R, L, C):
    return np.atan(((1/(2*np.pi*f*C))-(2*np.pi*f*L))/(R))
p0_phi = [30, 113*1e-3, 217*1e-9]
popt_phi, pcov_phi = curve_fit(model_phi, rez_f, rez_phi_g, p0_phi)
G_omega_0 = np.max(model_G(linspace_f, *popt_G))
G_fwhm = G_omega_0/(2**0.5)
rez_x = f0.n

def inverse_f_minus(G):
    A2 = L.n**2
    B2 = (R.n**2 - 2*(L.n/C.n) - 1/(G**2))
    C2 = 1/(C.n**2)
    t = ((-1)*B2 - (B2**2 - 4*A2*C2)**0.5)/(2*A2)
    return (np.sqrt(t))/(2*np.pi)
f_1 = inverse_f_minus(G_fwhm)

def inverse_f_plus(G):
    A2 = L.n**2
    B2 = (R.n**2 - 2*(L.n/C.n) - 1/(G**2))
    C2 = 1/(C.n**2)
    t = ((-1)*B2 + (B2**2 - 4*A2*C2)**0.5)/(2*A2)
    return (np.sqrt(t))/(2*np.pi)
f_2 = inverse_f_plus(G_fwhm)


fig, ax = plt.subplots(2, 1, figsize=(9,16), sharex=True)
ax[0].set_ylabel(r"$|\hat{G}|$ [mS]", fontsize=15)
ax[0].scatter(rez_f, rez_g*1e3, marker='.', s=100, label='Naměřené hodnoty', color='blue')
ax[0].plot(linspace_f, model_G(linspace_f, *popt_G)*1e3, label="Fit", color='red')
ax[0].axvline(x=rez_x, color='k', linestyle='-.', linewidth=1.5, label=f'Rezonanční frekvence:\n{rez_x:.2f} Hz')
ax[0].axhline(y=(G_fwhm)*1e3, color='g', linestyle='--', linewidth=1.5, label=r'$|\hat{{G_0}}|/\sqrt{2}$')

ax[0].plot([f_1, f_2], [(G_fwhm)*1e3, (G_fwhm)*1e3], linestyle='-', linewidth=1.5, color='orange', label=r'$2 \alpha$')
ax[0].scatter(f_1, G_fwhm*1e3, marker='<', color='orange')
ax[0].scatter(f_2, G_fwhm*1e3, marker='>', color='orange')
ax[0].tick_params(labelsize=10)
ax[0].grid(True, alpha=0.7)
ax[0].legend(fontsize=10)

ax[1].set_xlabel(r"$f$ [Hz]", fontsize=15)
ax[1].set_ylabel(r"$\varphi$ [°]", fontsize=15)
ax[1].scatter(rez_f, rez_phi_g*(180/np.pi), marker='.', s=100, label='Naměřené hodnoty', color='blue')
ax[1].plot(linspace_f, model_phi(linspace_f, *popt_phi)*(180/np.pi), label="Fit", color='red')
ax[1].axvline(x=rez_x, color='k', linestyle='-.', linewidth=1.5, label=f'Rezonanční frekvence:\n{rez_x:.2f} Hz')
ax[1].tick_params(labelsize=10)
ax[1].grid(True, alpha=0.7)
ax[1].legend(fontsize=10)
fig.subplots_adjust(hspace=0.0)
path_vodivost = downloads_dir / 'vodivost.pdf'
plt.savefig(path_vodivost, dpi=300, bbox_inches='tight')

In [ ]:
# nacteni google tabulek
sheet_id = '1UsP814IJrfRsE0t9Es3ti1vHPey5zNOwflnBbbTUxkY'
gid = '1913907669'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_prechod = pd.read_csv(url)

In [ ]:
# prechodovy jev

podcas = np.array(df_prechod['podcas'].iloc[150:1500])
pod1 = np.array(df_prechod['pod1'].iloc[150:1500])

def model_pod(t, Uc, alpha, omegad, phi, Uf):
    return Uc*np.exp(-alpha*t)*np.cos(omegad*t + phi) + Uf
p0_pod = [-0.1, 445, 6351, -1, -0.98]
popt_pod, pcov_pod = curve_fit(model_pod, podcas, pod1, p0_pod)
Uc, alpha, omegad, phi, Uf = popt_pod
_, alpha_u, omegad_u, _,_ = np.sqrt(np.diag(pcov_pod))
alpha = uf(alpha, alpha_u)
omegad = uf(omegad, omegad_u)
pod_R = 2*alpha*L +10.4
pod_omega = (omegad**2 + alpha**2)**0.5
pod_f0 = pod_omega/(2*np.pi)
print(f'alpha = {alpha:.1uPL}\nomegad = {omegad:.1uPL}\npod_R = {pod_R:.1uPL}\npod_omega = {pod_omega:.1uPL}\npod_f0 = {pod_f0:.1uPL}')


fig, ax = plt.subplots(figsize=(16,9))
ax.set_ylabel(fr"$U\,[V]$", fontsize=20)
ax.set_xlabel(fr"$t\,[s]$", fontsize=20)
ax.plot(podcas, pod1, label='Hodnoty', color='blue')
ax.plot(podcas, model_pod(podcas, *popt_pod), label='Fit', color='red', ls='--')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
path_pod = downloads_dir / 'podkriticke.pdf'
plt.savefig(path_pod, dpi=300, bbox_inches='tight')

In [ ]:
# kriticke

kritcas = np.array(df_prechod['kritcas'].iloc[170:400])
krit1 = np.array(df_prechod['krit1'].iloc[170:400])

def model_krit(t, Uc6, Uc7, lambd, Uf):
    return (Uc6 + Uc7*t)*np.exp(lambd*t)+Uf
p0_krit = [0.2, -2182, -8118, 0.98]
popt_krit, pcov_krit = curve_fit(model_krit, kritcas, krit1, p0_krit)
print(popt_krit)

fig, ax = plt.subplots(figsize=(16,9))
ax.set_ylabel(fr"$U\,[V]$", fontsize=20)
ax.set_xlabel(fr"$t\,[s]$", fontsize=20)
ax.plot(kritcas, krit1, label='Hodnoty', color='blue')
#ax.plot(kritcas, model_krit(kritcas, *popt_krit), label='Fit', color='red', ls='--')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
path_krit = downloads_dir / 'kriticke.pdf'
plt.savefig(path_krit, dpi=300, bbox_inches='tight')

print(2*L*omega0)

In [ ]:
# nadkriticke

nadcas = np.array(df_prechod['nadcas'].iloc[170:400])
nad1 = np.array(df_prechod['nad1'].iloc[170:400])

def model_nad(t, Uc1, lambd1, Uf):
    return Uc1*np.exp(lambd1*t)  + Uf
p0_nad = [1, -1000, 1]
popt_nad, pcov_nad = curve_fit(model_nad, nadcas, nad1, p0_nad)
print(popt_nad)

fig, ax = plt.subplots(figsize=(16,9))
ax.set_ylabel(fr'$U\,[V]$', fontsize=20)
ax.set_xlabel(fr'$t\,[s]$', fontsize=20)
ax.plot(nadcas, nad1, label='Hodnoty', color='blue')
ax.plot(nadcas, model_nad(nadcas, *popt_nad), label='Fit', color='red', ls='--')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
path_nad = downloads_dir / 'nadkriticke.pdf'
plt.savefig(path_nad, dpi=300, bbox_inches='tight')

In [ ]:
logUc = np.log(-nad1+9.85057989e-01)
def func(t, A, B):
    return A*t+B
popt, pcov = curve_fit(func, nadcas, logUc)
A= uf(popt[0], np.sqrt(pcov[0,0]))
B= uf(popt[1], np.sqrt(pcov[1,1]))
print(f"A = {A}, B = {B}")
R=-(L)*((2*np.pi*rez_x)**2+A**2)/A
print(f"R = {R}")

fig, ax = plt.subplots(figsize=(16,9))
ax.set_ylabel(fr"$ln(U_f-U_c)$", fontsize=20)
ax.set_xlabel(fr"$t\,[s]$", fontsize=20)
ax.scatter(nadcas, logUc, label='Hodnoty', color='blue', s=20)
ax.plot(nadcas, func(nadcas, *popt), label='Fit', color='red', ls='--')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
path_ln = downloads_dir / 'nadkriticke_ln.pdf'
plt.savefig(path_ln, dpi=300, bbox_inches='tight')


fig, ax = plt.subplots(figsize=(16,9))
ax.set_ylabel(fr"$U\,[V]$", fontsize=20)
ax.set_xlabel(fr"$t\,[s]$", fontsize=20)
ax.scatter(nadcas, nad1, label='Hodnoty', color='blue', s=20)
#ax.plot(nadcas, 9.85057989e-01 - np.exp(func(nadcas, *popt)), label='Fit', color='red', ls='--')
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.7)
ax.legend(fontsize=15)
path_e = downloads_dir / 'nadkriticke_e.pdf'
plt.savefig(path_e, dpi=300, bbox_inches='tight')

R_final = 50 + 10.4 + 3320+ 3 + 16
print(f"R = {R_final} ")